## 1. Problem Definition

**Tree-based models are highly effective for fraud detection because they can:**

* Learn non-linear relationships
* Handle feature interactions automatically
* Capture threshold-based fraud behavior
* Work well with imbalanced tabular datasets
* Provide feature importance explanations

**In this notebook, we explore:**

1. Decision Tree Classifier
2. Random Forest Classifier

**We evaluate them using fraud-focused metrics:**

* Recall
* F1-Score
* ROC-AUC
* Confusion Matrix

## 2. Mathematical Intuition

**Decision Trees:**

A Decision Tree recursively splits data using feature thresholds that maximize class separation.

**Common split criteria:**

**Gini Impurity**

Gini = 1 - \sum p_i^2

**Entropy**

Entropy = -\sum p_i \log_2(p_i)

The tree selects the split with maximum information gain.

**Random Forest:**

**Random Forest improves Decision Trees by:**

* Training multiple trees
* Using bootstrap sampling
* Using random feature subsets

**Final prediction:**

$$\hat{y} = \text{Majority Vote of Trees}$$

**Advantages:**

* Lower variance
* Better generalization
* More robust to noise
* Strong fraud detection performance

## 3. Bias vs Variance

| Model | Bias | Variance |
| :--- | :--- | :--- |
| **Decision Tree** | Low | High |
| **Random Forest** | Low | Low |

**Key Insight**

* Decision Trees often overfit
* Random Forest stabilizes predictions through ensemble averaging

In [ ]:
# Setup
import sys
sys.path.append("..")

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)

# Project Modules
from src.data_loader import load_raw_data
from src.preprocess import preprocess_dataset
from src.config import RANDOM_STATE

# Styling
sns.set_style("whitegrid")

## 5. Load & Preprocess Data

In [ ]:
# Load raw dataset
df = load_raw_data()

# Preprocess dataset
X_train, X_test, y_train, y_test, preprocessor = preprocess_dataset(df)

## 6. Decision Tree Model

**Initialize Model**

In [ ]:
decision_tree = DecisionTreeClassifier(
    max_depth=6,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Fit model
decision_tree.fit(X_train, y_train)

# Predictions
y_pred_tree = decision_tree.predict(X_test)

y_pred_tree_proba = decision_tree.predict_proba(X_test)[:, 1]

## 7. Decision Tree Evaluation

**Classification Report**

In [ ]:
print(classification_report(y_test, y_pred_tree))

**ROC-AUC Score**

In [ ]:
roc_auc_tree = roc_auc_score(
    y_test,
    y_pred_tree_proba
)

print(f"Decision Tree ROC-AUC: {roc_auc_tree:.4f}")

**Confusion Matrix**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_tree,
    cmap="Blues",
    ax=ax
)

plt.title("Decision Tree Confusion Matrix")
plt.show()

**ROC Curve**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_tree_proba,
    ax=ax
)

plt.title("Decision Tree ROC Curve")
plt.show()

## 8. Random Forest Model

**Initialize Mode**

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

**Train Model**

In [ ]:
random_forest.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_rf = random_forest.predict(X_test)

y_pred_rf_proba = random_forest.predict_proba(X_test)[:, 1]

## 9. Random Forest Evaluation

**Classification Report**

In [ ]:
print(classification_report(y_test, y_pred_rf))

**ROC-AUC Score**

In [ ]:
roc_auc_rf = roc_auc_score(
    y_test,
    y_pred_rf_proba
)

print(f"Random Forest ROC-AUC: {roc_auc_rf:.4f}")

**Confusion Matrix**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_rf,
    cmap="Greens",
    ax=ax
)

plt.title("Random Forest Confusion Matrix")
plt.show()

**ROC Curve**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_rf_proba,
    ax=ax
)

plt.title("Random Forest ROC Curve")
plt.show()

## 10. Feature Importance Analysis

Tree models provide intrinsic feature importance scores.

**Random Forest Feature Importance**

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": random_forest.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

feature_importance.head(15)

**Visualization**

In [ ]:
plt.figure(figsize=(10, 8))

sns.barplot(
    data=feature_importance.head(15),
    x="importance",
    y="feature"
)

plt.title("Top 15 Important Features")
plt.xlabel("Importance Score")
plt.ylabel("Feature")

plt.show()

## Conclusion

The Random Forest model outperformed the standalone Decision Tree and slightly improved upon Logistic Regression by achieving stronger fraud precision and F1-score while maintaining high recall.

Feature importance analysis revealed that anomaly-driven behavioral indicators dominate fraud prediction performance, particularly:

* anomaly_score
* device_anomaly_interaction
* composite_risk_score

This confirms that engineered interaction features significantly improved the model’s ability to capture complex fraud patterns beyond simple linear relationships.

The Random Forest model demonstrates strong suitability for enterprise fraud detection due to:

* high fraud recall
* strong class separation
* robustness to noisy behavioral data
* interpretable feature importance rankings

The model will serve as a strong benchmark for subsequent boosting and ensemble experiments.